# Spam2000 Reconstruction

Optional technical review notebook. The production CLI workflow lives in `../scripts/`.


# 03 | Audit SPAM2000 crop completion and correction provenance

**FABLE historical crop-area reconstruction · Scientific review stage**

This notebook reviews the **existing** SPAM2000 completion outputs against the frozen 56-product reference and corrected native SPAM files. It does not invent missing data or overwrite any of the accepted mapping assets.

**Processing context:** Notebook 01 standardizes the native 5′ SPAM releases. Notebook 02 freezes the SPAM–FABLE mapping weights. The earlier development notebook (`02_freeze_and_validate_SPAM_FABLE_mapping_FINAL`, Cell 3S) produced two distinct 2000 country-level outputs: a scientific completion layer and a Calculator operational layer. Their treatment of unsupported records differs intentionally.

**Important methodological boundary:** This is an **audit and release gate**, not a replacement for the original reconstruction computations. The source-based construction, approved 23 MIRCA share corrections, OOIL mapping precedence, and final grid-level integration must be independently verified before publication.

## 01 | Environment and source locations

The repository supports local overrides via `SPAM_PROJECT_ROOT`. No GUI popup is required. Inputs are read-only; audit outputs are saved in a dedicated directory.

In [ ]:
# 01 | Paths, dependencies, and frozen inputs

from pathlib import Path
import os
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("SPAM_PROJECT_ROOT", r"PROJECT_ROOT\WORK")).expanduser()
DATA_ROOT = PROJECT_ROOT / "Data"
RELEASE_ROOT = DATA_ROOT / "SPAM_FABLE_FINAL"
NATIVE_COUNTRY = RELEASE_ROOT / "01_native_5arcmin" / "01_SPAM_native_country_totals_QC.csv.gz"
PRODUCT_REFERENCE = RELEASE_ROOT / "02_mapping" / "02_FABLE_56_product_reference.csv"
SCIENTIFIC_FILE = RELEASE_ROOT / "02_country_completion" / "SPAM2000_FABLE56_country_SCIENTIFIC_completion.csv.gz"
OPERATIONAL_FILE = RELEASE_ROOT / "02_country_completion" / "SPAM2000_FABLE56_country_CALCULATOR_operational.csv.gz"
AUDIT_DIR = RELEASE_ROOT / "03_reconstruction_audit"

inputs = {"native_country": NATIVE_COUNTRY, "products": PRODUCT_REFERENCE,
          "scientific_completion": SCIENTIFIC_FILE, "calculator_completion": OPERATIONAL_FILE}
source_inventory = pd.DataFrame([
    {"asset": name, "path": str(path), "available": path.is_file(),
     "size_bytes": path.stat().st_size if path.is_file() else pd.NA}
    for name, path in inputs.items()
])
display(source_inventory)
missing = source_inventory.loc[~source_inventory["available"], "asset"].tolist()
if missing:
    raise FileNotFoundError(
        "Missing prerequisite assets: " + ", ".join(missing) +
        ". Recover/rebuild the existing 2000 completion using the archived Cell 3S "
        "before running this audit. No zero-filled replacement will be created."
    )
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("SPAM2000 reconstruction audit inputs available.")

## 02 | Read and verify the historical panel

The expected Cell 3S panel is **177 countries × 56 FABLE products = 9,912 records per layer** for year 2000. We verify keys and product support before testing area values.

In [ ]:
# 02 | Country × product structure and source identifiers

scientific = pd.read_csv(SCIENTIFIC_FILE, low_memory=False)
operational = pd.read_csv(OPERATIONAL_FILE, low_memory=False)
products = pd.read_csv(PRODUCT_REFERENCE, low_memory=False)
native = pd.read_csv(NATIVE_COUNTRY, low_memory=False)

keys = ["country_iso3", "fable_product"]
area = ["harvested_irrigated_ha", "harvested_rainfed_ha", "harvested_total_ha",
        "physical_irrigated_ha", "physical_rainfed_ha", "physical_total_ha"]
for title, frame in [("scientific", scientific), ("operational", operational)]:
    required = set(keys + area)
    missing_cols = sorted(required - set(frame.columns))
    if missing_cols:
        raise ValueError(f"{title} lacks required columns: {missing_cols}")
    if frame.duplicated(keys).any():
        raise ValueError(f"{title} contains duplicate country–product keys")
    if "year" in frame and not frame["year"].eq(2000).all():
        raise ValueError(f"{title} includes years outside 2000")

expected_products = set(products["fable_product"].astype(str))
for title, frame in [("scientific", scientific), ("operational", operational)]:
    unknown = set(frame["fable_product"].astype(str)) - expected_products
    if unknown: raise ValueError(f"Unexpected FABLE products in {title}: {sorted(unknown)}")
if set(map(tuple, scientific[keys].to_numpy())) != set(map(tuple, operational[keys].to_numpy())):
    raise ValueError("Scientific and operational layers have different keys")

panel = pd.DataFrame([
    {"layer": name, "rows": len(df), "countries": df.country_iso3.nunique(),
     "products": df.fable_product.nunique(), "duplicate_keys": int(df.duplicated(keys).sum())}
    for name, df in [("scientific", scientific), ("operational", operational)]
])
display(panel)
if len(expected_products) != 56 or not ((panel.rows == 9912) & (panel.countries == 177) & (panel.products == 56)).all():
    raise AssertionError("2000 product/country coverage differs from archived Cell 3S expectations; investigate")

## 03 | Scientific versus operational values

A missing scientific observation is not equivalent to a zero. Where the operational layer substitutes explicit modeled zeros, the scientific layer must retain missingness and the methodological provenance must identify the assumption. We report differences rather than silently adopting the Calculator layer as observed data.

In [ ]:
# 03 | Compare layer values and missingness

for frame in (scientific, operational):
    for col in area:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")

joined = scientific[keys+area].merge(
    operational[keys+area], on=keys, suffixes=("_sci", "_calc"), validate="one_to_one"
)
missing_scientific = joined[[f"{col}_sci" for col in area]].isna().any(axis=1)
missing_operational = joined[[f"{col}_calc" for col in area]].isna().any(axis=1)
diff_flags = pd.DataFrame({
    col: ~(
        np.isclose(joined[f"{col}_sci"], joined[f"{col}_calc"], rtol=1e-10, atol=1e-6, equal_nan=True)
    ) for col in area
})
joined["any_layer_difference"] = diff_flags.any(axis=1)
joined["scientific_missing_component"] = missing_scientific
joined["operational_missing_component"] = missing_operational

summary = pd.DataFrame([{
    "scientific_rows_with_missing_components": int(missing_scientific.sum()),
    "operational_rows_with_missing_components": int(missing_operational.sum()),
    "rows_with_different_area_components": int(joined.any_layer_difference.sum()),
}])
display(summary)
display(joined.loc[joined.any_layer_difference, keys + ["scientific_missing_component", "operational_missing_component"]].head(35))

# The old scientific layer intentionally contained 26 unresolved rows.
# Avoid interpreting this general component-missing count as a direct reproduction
# of the historical 26-row release gate: physical splits can be source-unavailable.


## 04 | Area identities and unavailable physical components

Conservation is checked only where all required components are observed or reconstructed. For the source-total-only SPAM2000 OTHE records, an unavailable physical IR/RF split must **not** be converted into a numerical zero.

In [ ]:
# 04 | Numerical area consistency, non-negativity, and source-only OTHE

def area_qc(name, frame):
    rows=[]
    for prefix in ("harvested", "physical"):
        a=frame[f"{prefix}_irrigated_ha"]
        b=frame[f"{prefix}_rainfed_ha"]
        total=frame[f"{prefix}_total_ha"]
        complete=a.notna() & b.notna() & total.notna()
        residual=(a+b-total).abs().where(complete)
        rows.append({"layer":name, "component":prefix, "complete_rows":int(complete.sum()),
                     "max_abs_balance_error_ha": float(residual.max()) if complete.any() else np.nan,
                     "negative_values":int((frame[[f"{prefix}_irrigated_ha", f"{prefix}_rainfed_ha", f"{prefix}_total_ha"]] < -1e-8).sum().sum())})
    return rows

area_checks = pd.DataFrame(area_qc("scientific", scientific)+area_qc("operational", operational))
display(area_checks)
if (area_checks["negative_values"] > 0).any() or (area_checks["max_abs_balance_error_ha"].dropna() > 1e-5).any():
    raise AssertionError("Area accounting failed; inspect source and reconstructed values")

source_othe = native.loc[(native["year"]==2000) & (native["source_product_code"].astype(str).str.lower()=="othe")].copy()
source_othe["physical_total_ha"] = pd.to_numeric(source_othe["physical_total_ha"], errors="coerce")
othe_info = {
 "countries":int(source_othe.country_iso3.nunique()),
 "physical_total_ha":float(source_othe.physical_total_ha.sum()),
 "physical_irrigated_known":int(source_othe.physical_irrigated_ha.notna().sum()),
 "physical_rainfed_known":int(source_othe.physical_rainfed_ha.notna().sum())}
print("Native source OTHE:", othe_info)
if othe_info["countries"]!=176 or not np.isclose(othe_info["physical_total_ha"], 559861492.7,atol=0.01):
    raise AssertionError("Notebook 01 corrected OTHE country handoff is inconsistent")

## 05 | Provenance inventory and correction gate

The legacy notebooks document a **23-key MIRCA-informed irrigation-share harmonization** and a separate SPAM2000 completion exercise. They are different corrections. The number 23 is **not** evidence that the published candidate automatically contains those corrections.

This stage records available provenance columns and does **not** certify approved 23-key application without a machine-readable, source-keyed correction audit and a comparison against the accepted pre-correction layer.

In [ ]:
# 05 | Record provenance columns and outstanding correction checks

search_terms = ("source", "method", "status", "provenance", "confidence", "resolution", "correction", "observed", "model")
prov_inventory = pd.DataFrame([
    {"layer":name,"column":col}
    for name,df in [("scientific",scientific),("operational",operational)]
    for col in df.columns if any(term in col.lower() for term in search_terms)
])
display(prov_inventory)

gates = pd.DataFrame([
    {"gate":"Historical country-product panel", "status":"CHECKED"},
    {"gate":"Layer differences and missing-data semantics", "status":"CHECKED"},
    {"gate":"Area component conservation", "status":"CHECKED"},
    {"gate":"OTHE source physical-total preservation", "status":"CHECKED"},
    {"gate":"Approved 23 MIRCA keys and correction precedence", "status":"PENDING_SOURCE_KEY_AUDIT"},
    {"gate":"OOIL mapping candidate accepted in integrated 2000 output", "status":"PENDING_LINEAGE_COMPARISON"},
    {"gate":"2000–2005/2010/2015/2020 non-target year identity", "status":"PENDING_HISTORICAL_INTEGRATION"},
    {"gate":"Clean start reproduction of Cell 3S source calculations", "status":"PENDING_RECONSTRUCTION_RUN"},
])
display(gates)


## 06 | Export reproducible audit evidence

Exports are clearly labeled as **audit outputs**, never final reconstructed SPAM production data. The review remains open until the correction lineage and fresh-run gates above are resolved.

In [ ]:
# 06 | Write QC outputs and input hashes

def sha256(path, block=1024*1024):
    digest=hashlib.sha256()
    with open(path,"rb") as stream:
        for piece in iter(lambda: stream.read(block),b""):
            digest.update(piece)
    return digest.hexdigest()

panel.to_csv(AUDIT_DIR/"03_panel_structure_qc.csv",index=False)
summary.to_csv(AUDIT_DIR/"03_scientific_operational_comparison.csv",index=False)
area_checks.to_csv(AUDIT_DIR/"03_area_balance_qc.csv",index=False)
prov_inventory.to_csv(AUDIT_DIR/"03_provenance_columns.csv",index=False)
gates.to_csv(AUDIT_DIR/"03_release_gates.csv",index=False)
joined.loc[joined.any_layer_difference, keys + ["scientific_missing_component", "operational_missing_component"]].to_csv(
    AUDIT_DIR/"03_country_product_layer_differences.csv",index=False)
manifest={"stage":"SPAM2000 reconstruction audit", "status":"AUDIT_COMPLETE_CORRECTION_LINEAGE_PENDING",
          "inputs":{name:{"path":str(path),"sha256":sha256(path)} for name,path in inputs.items()},
          "ot he_source_total_ha".replace(" ",""):othe_info["physical_total_ha"],
          "correction_gates":gates.to_dict(orient="records")}
with open(AUDIT_DIR/"03_audit_manifest.json","w",encoding="utf-8") as file:
    json.dump(manifest,file,indent=2)
print("Audit outputs:", AUDIT_DIR)
print("STATUS: AUDIT_COMPLETE_CORRECTION_LINEAGE_PENDING")

## Interpretation and next processing stage

Passing the audit confirms that the existing country-level completion layers have the expected structure, traceable differences, consistent area arithmetic and corrected source OTHE total. **It does not certify the reconstruction methodology or the final historical dataset.**

Before a publication release, the original Cell 3S reconstruction must be reorganized as a self-contained production stage, its scientific assumptions reviewed, and the approved MIRCA correction keys integrated with documented precedence. Grid-level reconstruction and other-year preservation belong to the subsequent observed-year and historical stages. The native OTHE source-only physical total remains available as source context and must never be silently allocated across FABLE crops.

**Planned primary outputs of the eventual production stage:** scientific 2000 completion, Calculator operational 2000 layer, per-record provenance and 23-key correction audit. This notebook only creates validation reports.

## 07 | Identify archived correction and integrated-release evidence

The earlier workflow contains **two different MIRCA-supported operations**: (i) irrigation-share harmonization of 23 selected country–product records; and (ii) a separate 23-record same-year MIRCA *magnitude fallback* during missing-product completion. The identical counts do **not** make these the same records. This audit does not apply either operation again.

Set `LEGACY_HARMONIZATION_DIR` to the exact `06_harmonized_irrigation` output folder if the automatic candidates do not resolve. Files are inspected read-only. Missing archival sources are reported as `SOURCE_UNAVAILABLE`, **not** as passed validations.

In [ ]:
# 07 | Locate original harmonization and historical releases

# Edit these two optional paths only if archived files are elsewhere.
LEGACY_HARMONIZATION_DIR = Path(os.environ.get(
    "SPAM_LEGACY_HARMONIZATION_DIR",
    r"PROJECT_ROOT\Python Scripts\FINAL SPAM SCRIPt\data\work\SPAMFABLE_harmonized_irrigation"
))

archive_paths = {
    "legacy_provenance": LEGACY_HARMONIZATION_DIR / "06_SPAM2000_harmonization_provenance.csv",
    "legacy_harmonized_2000": LEGACY_HARMONIZATION_DIR / "06_SPAM2000_harmonized_H_P_intensity_layer.csv.gz",
    "legacy_historical": LEGACY_HARMONIZATION_DIR / "06_FABLE_C_SPAM_historical_input_2000_2020_HARMONIZED.csv.gz",
    "rebuilt_historical": RELEASE_ROOT / "04_final_release" / "calculator" /
        "04_FABLE_C_SPAM_historical_input_2000_2020_REBUILT_2000.csv.gz",
    "final_historical": RELEASE_ROOT / "04_final_release" / "calculator" /
        "04_FABLE_C_SPAM_historical_input_2000_2020_FINAL.csv.gz",
}

archive_inventory = pd.DataFrame([
    {"asset": k, "path": str(v), "available": v.is_file(),
     "size_bytes": v.stat().st_size if v.is_file() else pd.NA}
    for k, v in archive_paths.items()
])
display(archive_inventory)
print("Missing archive assets are not recreated or inferred.")

## 08 | Verify the archived MIRCA irrigation-share correction keys

This step audits the **identity and provenance** of the approved harmonization records. It does not assume that a legacy corrected row must equal a later country-level reconstructed product: differences in production branch, parent aggregation or area magnitude require explicit reconciliation. If a comparable 2000 legacy layer is available, the source key set is compared against the archived harmonization output; the modern scientific layer is then checked for coverage and reported component differences. Differences remain `REVIEW`, never silently overwritten.

In [ ]:
# 08 | Original 23-key irrigation-share correction evidence

share_key_audit = pd.DataFrame()
share_summary = {"gate": "MIRCA_IRRIGATION_SHARE_23_KEYS", "status": "SOURCE_UNAVAILABLE"}
keycols = ["country_iso3", "fable_product"]
legacy_provenance_path = archive_paths["legacy_provenance"]

if legacy_provenance_path.is_file():
    legacy_prov = pd.read_csv(legacy_provenance_path, low_memory=False)
    if not set(keycols).issubset(legacy_prov.columns):
        raise ValueError(f"Legacy correction provenance lacks {keycols}")
    prov_keys = legacy_prov.loc[:, keycols].drop_duplicates()
    prov_duplicates = int(legacy_prov.duplicated(keycols).sum())
    if prov_duplicates:
        raise AssertionError("Duplicate original MIRCA share-correction keys")
    share_key_audit = prov_keys.merge(
        scientific[keycols + ["harvested_irrigated_ha", "harvested_rainfed_ha", "harvested_total_ha"]],
        on=keycols, how="left", indicator=True, validate="one_to_one"
    )
    share_summary = {
        "gate": "MIRCA_IRRIGATION_SHARE_23_KEYS",
        "original_keys": len(prov_keys),
        "matched_modern_2000_keys": int(share_key_audit["_merge"].eq("both").sum()),
        "missing_modern_keys": int(share_key_audit["_merge"].ne("both").sum()),
        "status": "KEYS_PRESENT_LINEAGE_REVIEW" if len(prov_keys)==23 and share_key_audit["_merge"].eq("both").all() else "REVIEW_KEYSET"
    }
    if archive_paths["legacy_harmonized_2000"].is_file():
        legacy_2000 = pd.read_csv(archive_paths["legacy_harmonized_2000"], low_memory=False)
        if set(keycols + ["harvested_irrigated_ha", "harvested_rainfed_ha"]).issubset(legacy_2000.columns):
            legacy_2000 = legacy_2000.loc[:, keycols + ["harvested_irrigated_ha", "harvested_rainfed_ha"]]
            if legacy_2000.duplicated(keycols).any():
                raise AssertionError("Legacy 2000 reference contains duplicate country-product keys")
            compare = prov_keys.merge(legacy_2000, on=keycols, how="left", validate="one_to_one")
            compare = compare.merge(
                scientific[keycols + ["harvested_irrigated_ha", "harvested_rainfed_ha"]],
                on=keycols, suffixes=("_legacy", "_modern"), validate="one_to_one"
            )
            for component in ["harvested_irrigated_ha", "harvested_rainfed_ha"]:
                compare[component + "_abs_difference_ha"] = (
                    compare[component + "_modern"] - compare[component + "_legacy"]
                ).abs()
            share_key_audit = compare
            share_summary["max_irrigated_diff_ha"] = float(compare["harvested_irrigated_ha_abs_difference_ha"].max())
            share_summary["max_rainfed_diff_ha"] = float(compare["harvested_rainfed_ha_abs_difference_ha"].max())
            share_summary["status"] = "COMPARED_REVIEW_DIFFERENCES_AND_PRECEDENCE"
else:
    print("Legacy provenance not found. Set SPAM_LEGACY_HARMONIZATION_DIR to the archived output directory.")

display(pd.DataFrame([share_summary]))
if not share_key_audit.empty:
    display(share_key_audit.head(30))

## 09 | Verify OOIL provenance and 2000 product-area accounting

The modern completion already contains country-level FABLE products. This check identifies oilseed-related records and their provenance fields, but **cannot prove the revised OOIL parent-to-child weights were used** unless the frozen parent-weight audit and the specific prior candidate are compared. Consequently an area-balance pass alone does not close the OOIL lineage gate.

In [ ]:
# 09 | Inspect OOIL-linked modern reconstruction evidence

ooil_products = ["oilseed_other", "oilpalmfruit", "rapeseed", "sesame", "sunflower", "olive", "groundnut", "coconut", "soyabean"]
ooil_rows = scientific.loc[scientific.fable_product.isin(ooil_products)].copy()
provenance_cols = [c for c in ["harvested_magnitude_source", "IR_RF_structure_source", "physical_area_source",
                                "resolution_group", "resolution_confidence", "observed_SPAM"] if c in ooil_rows.columns]
ooil_provenance = (ooil_rows.groupby(["fable_product"] + provenance_cols, dropna=False)
                   .agg(records=("country_iso3", "size"), harvested_total_ha=("harvested_total_ha", "sum"))
                   .reset_index())
display(ooil_provenance.head(50))

ooil_lineage = {"gate": "OOIL_REVISED_MAPPING_LINEAGE", "oil_related_rows": len(ooil_rows),
                "status": "PENDING_PARENT_CHILD_SOURCE_COMPARISON"}
print("A definitive OOIL pass requires the frozen 2000 OOIL parent-child allocation audit, not only this completed panel.")

## 10 | Historical non-target-year identity

Compare the archived integrated `REBUILT_2000` file against the post-repair `FINAL` export using the **same schema and stable country–year–product keys**. A failure or missing source is not converted into a pass. This proves equality between these *two versions* only; the original pre-repair baseline needs a separate source comparison if strict non-2000 preservation is to be claimed.

In [ ]:
# 10 | Compare unchanged historical years across release files

historical_identity = {"gate":"NON_2000_RELEASE_IDENTITY", "status":"SOURCE_UNAVAILABLE"}
historical_differences = pd.DataFrame()
path_rebuilt = archive_paths["rebuilt_historical"]
path_final = archive_paths["final_historical"]

if path_rebuilt.is_file() and path_final.is_file():
    rebuilt = pd.read_csv(path_rebuilt, low_memory=False)
    final = pd.read_csv(path_final, low_memory=False)
    common_keys = [c for c in ["country_iso3", "year", "fable_product"] if c in rebuilt.columns and c in final.columns]
    if len(common_keys) != 3:
        raise ValueError("Historical files need country_iso3/year/fable_product keys for exact reconciliation")
    for label, frame in [("rebuilt", rebuilt),("final", final)]:
        if frame.duplicated(common_keys).any():
            raise AssertionError(f"{label} contains duplicate historical country-year-product keys")
    old = rebuilt.loc[rebuilt.year.isin([2005,2010,2015,2020])]
    new = final.loc[final.year.isin([2005,2010,2015,2020])]
    shared = [c for c in old.columns if c in new.columns and c not in common_keys]
    if set(old.columns) != set(new.columns):
        historical_identity["status"] = "REVIEW_SCHEMA_DIFFERENCE"
    else:
        comp = old[common_keys + shared].merge(new[common_keys + shared], on=common_keys,
                                                how="outer", indicator=True, suffixes=("_rebuilt","_final"),
                                                validate="one_to_one")
        absent = int(comp["_merge"].ne("both").sum())
        row_diff = np.zeros(len(comp),dtype=bool)
        for col in shared:
            a,b=comp[col+"_rebuilt"],comp[col+"_final"]
            if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
                row_diff |= ~np.isclose(a,b,atol=1e-7,rtol=1e-12,equal_nan=True)
            else:
                row_diff |= ~(a.fillna("<missing>").astype(str).eq(b.fillna("<missing>").astype(str))).to_numpy()
        historical_differences = comp.loc[row_diff | comp["_merge"].ne("both"),common_keys+["_merge"]].copy()
        historical_identity.update({"rows_rebuilt":len(old), "rows_final":len(new),
                                    "unmatched_keys":absent,"different_rows":len(historical_differences),
                                    "status":"PASS_PAIRWISE_IDENTITY" if len(historical_differences)==0 else "REVIEW_DIFFERENCES"})

display(pd.DataFrame([historical_identity]))
if not historical_differences.empty:
    display(historical_differences.head(30))

## 11 | Persist correction-lineage gates (read-only audit)

This audit cannot substitute for a clean rebuild of archived Cell 3S or a direct 2000 parent-to-child mapping trace. Archive status therefore remains explicitly pending until each original calculation can be recreated from its source files. The legacy MIRCA share-correction audit and the MIRCA magnitude fallback remain distinct.

In [ ]:
# 11 | Save lineage report without changing scientific outputs

lineage_gates = pd.DataFrame([
    share_summary,
    ooil_lineage,
    historical_identity,
    {"gate":"MIRCA_MAGNITUDE_FALLBACK_23", "status":"PENDING_SEPARATE_CELL_3P_3S_KEY_AUDIT"},
    {"gate":"RECONSTRUCT_CELL_3S_CLEAN_RUN", "status":"PENDING_CLEAN_REBUILD"},
])
display(lineage_gates)
archive_inventory.to_csv(AUDIT_DIR/"03_archived_source_inventory.csv",index=False)
lineage_gates.to_csv(AUDIT_DIR/"03_correction_lineage_gates.csv",index=False)
if not share_key_audit.empty:
    share_key_audit.to_csv(AUDIT_DIR/"03_mirca_share_correction_comparison.csv",index=False)
ooil_provenance.to_csv(AUDIT_DIR/"03_ooil_provenance_inventory.csv",index=False)
if not historical_differences.empty:
    historical_differences.to_csv(AUDIT_DIR/"03_historical_release_differences.csv",index=False)
with open(AUDIT_DIR/"03_lineage_manifest.json","w",encoding="utf-8") as stream:
    json.dump({"status":"LINEAGE_AUDIT_REQUIRES_SOURCE_RECONCILIATION",
               "archive_inputs":{k:{"available":v.is_file(),"sha256":sha256(v) if v.is_file() else None} for k,v in archive_paths.items()},
               "gates":lineage_gates.where(pd.notna(lineage_gates),None).to_dict("records")},stream,indent=2,default=str)
print("Audit evidence saved:",AUDIT_DIR)
print("STATUS: LINEAGE_AUDIT_REQUIRES_SOURCE_RECONCILIATION")

## Publication decision

The reconstruction audit is a **read-only verification layer**, not the production reconstruction. A source-based clean rerun, explicit verification of the distinct 23-key MIRCA operations, OOIL parent allocation and historical baseline identity must pass before a definitive 2000 scientific release is declared. The 26 scientifically unresolved COM/PSE records remain missing unless independently supported by source evidence.